2.1 감성 분석 & 이벤트/투자의견 분류

In [ ]:
import sys, os

print('Python 버전:', sys.version.split()[0])
print('현재 디렉토리:', os.getcwd())

#OpenAI 라이브러리 설치 및 키 설정
# !pip install openai -q

from openai import OpenAI

#API Key 설정
os.environ['OPENAI_API_KEY'] = ''

#OpenAI 클라이언트 생성
client = OpenAI()

print('API 준비 완료')

Python 버전: 3.12.13
현재 디렉토리: /content
API 준비 완료


In [3]:
# 뉴스 1건을 감성 + 이벤트로 분석
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'))

news = 'C바이오, 신약 임상 3상 성공'

prompt = f'''금융 분석가로서 JSON 출력:
sentiment(긍정/부정/중립), event(이벤트유형)
뉴스: {news}
'''

r = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages=[{'role':'user','content':prompt}],
    response_format={'type':'json_object'})

print(r.choices[0].message.content)

{
  "sentiment": "긍정",
  "event": "신약 임상 성공"
}


In [ ]:
from openai import OpenAI

client = OpenAI(api_key='')

news = '회사가 적자 폭을 크게 줄였다'

prompt = f'''금융분석가로서 아래 뉴스의 투자 감성을 긍정/부정/중립 중 하나로만 답하라.(표현이 아닌 주가 관점)
뉴스 : {news}'''

r = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages = [{'role' : 'user',
                 'content' : prompt}])

print(r.choices[0].message.content)

긍정


In [5]:
# URL 을 사용: 크롤링

import json
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

client = OpenAI()

# 1. 뉴스 URL 입력
url = input("뉴스 URL을 입력하세요: ").strip()

# 2. 뉴스 제목 가져오기
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

title_tag = soup.select_one('meta[property="og:title"]')
news = title_tag["content"] if title_tag else soup.title.get_text(strip=True)

print("[수집된 뉴스 제목]")
print(news)

# 3. 뉴스 분석
prompt = f"""금융 분석가로서 다음 뉴스 제목을 분석하세요.

반드시 JSON 형식으로 출력하세요.
형식: {{"sentiment": "호재/악재/중립", "event": "이벤트유형", "reason": "판단 근거"}}

뉴스: {news}
"""

r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    response_format={"type": "json_object"}
)
# 4. 결과 출력
result = json.loads(r.choices[0].message.content)

print("\n[분석 결과]")
print(json.dumps(result, ensure_ascii=False, indent=2))

뉴스 URL을 입력하세요: https://www.ibtomato.com/ExternalView.aspx?type=1&no=15345
[수집된 뉴스 제목]
[IB토마토] 메리츠, 홈플러스 살리기에 손 내민 속내는

[분석 결과]
{
  "sentiment": "호재",
  "event": "기업 인수/합병",
  "reason": "메리츠가 홈플러스를 지원하는 것은 긍정적인 신호로, 기업의 재무적 안정성을 높일 수 있으며 향후 성장 가능성을 보여줍니다."
}


In [6]:
# [실습 3] 여러 리포트의 투자 의견을 분류하고 매수/중립/매도 개수를 집계하시오

from collections import Counter

reports = [
    'A전자 목표주가 상향, 비중 확대 추천',
    '실적 둔화 우려, 보수적 접근 권고',
    'HBM 성장 지속, 매수 의견 유지']

def rating(text):
  p = f'매수/중립/매도 중 하나로만 답: {text}'
  r = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages=[{'role':'user','content':p}])

  return r.choices[0].message.content.strip()

votes = Counter(rating(r) for r in reports)

print(votes)

Counter({'매수': 2, '매도': 1})


2.2 일반 모델 vs 금융 특화 모델

In [7]:
# FinBERT로 금융 문장 감성 분석하기

# 금융 특화 모델 FinBERT 사용 : pipeline(): 모델을 쉽게 불러와 바로 추론
from transformers import pipeline

finbert = pipeline('sentiment-analysis',
                    model='ProsusAI/finbert')

print(finbert('적자 폭을 크게 줄였다'))
# print(finbert("A기업은 3분기 적자 폭을 전년 대비 크게 줄이며 실적 개선세를 보였다."))
# print(finbert("A기업은 비용 절감과 매출 증가로 적자 폭을 크게 줄여 시장 기대를 상회했다."))
# print(finbert("A기업은 3분기 적자 폭을 전년 대비 크게 줄이며 실적 개선세를 보였다."))

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[{'label': 'neutral', 'score': 0.8964423537254333}]


In [8]:
# [실습 1] FinBERT로 금융 문장 여러 개의 감성을 분류

# !pip install transformers torch -q
from transformers import pipeline

finbert = pipeline('sentiment-analysis',
                   model = 'ProsusAI/finbert')

texts = [
    '회사가 적자 폭을 크게 줄였다',
    '예상보다 부진한 가이던스를 제시했다']

for t in texts:
  print(t, '->', finbert(t)[0])

회사가 적자 폭을 크게 줄였다 -> {'label': 'neutral', 'score': 0.8926175236701965}
예상보다 부진한 가이던스를 제시했다 -> {'label': 'neutral', 'score': 0.9037700295448303}


In [9]:
# 규칙 기반
def rule_sentiment(text):
  positive_words = [
      "증가", "개선", "흑자"]
  negative_words = [
      "감소", "실패", "적자"]
  if any(w in text for w in positive_words):
    return "긍정"

  if any(w in text for w in negative_words):
    return "부정"

  else:
    return "중립"

texts = [
    '회사가 적자 폭을 크게 줄였다',
    '예상보다 부진한 가이던스를 제시했다']

for t in texts:
  print(t, '->', rule_sentiment(t))

회사가 적자 폭을 크게 줄였다 -> 부정
예상보다 부진한 가이던스를 제시했다 -> 중립


In [10]:
# [실습 2] 같은 문장을 일반 LLM과 FinBERT로 분류해 결과 비교

text = '적자 폭 축소에도 투자심리는 위축'

prompt = f'''
다음 문장의 투자 감성을 긍정/부정/중립 중 하나로만 답하라.
문장:
{text}
'''

# 1) 일반 LLM
r = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}])
print('LLM   :', r.choices[0].message.content.strip())

# 2) 금융 특화 FinBERT로 감성 분석
print('FinBERT:', finbert(text)[0]['label'])

LLM   : 부정
FinBERT: neutral


In [11]:
# [실습 3] 여러 문장을 Finbert로 분석해 긍정비율을 계산하시오
# 한글과 영어의 성능 차이가 있음

finbert = pipeline('sentiment-analysis',
                   model = 'ProsusAI/finbert')

news = ['HBM 공급 계약 체결로 실적 기대',
        '환율 변동성 확대로 불확실성 증가',
        '신제품 흥행으로 매출 급증',
        "Expect performance from signing HBM supply contract",
        "Increasing uncertainty due to increased currency volatility",
        "Sales Surge Due to the Success of New Products"]

results = [finbert(n)[0]['label'] for n in news]
pos = results.count('positive')

print('결과 : ', results)
print(f'긍정 비율 : {pos}/{len(news)}')


결과 :  ['neutral', 'neutral', 'neutral', 'neutral', 'negative', 'positive']
긍정 비율 : 1/6


In [12]:
# 실수형으로 변경

from collections import Counter
import pandas as pd

news = ['HBM 공급 계약 체결로 실적 기대',
        '환율 변동성 확대로 불확실성 증가',
        '신제품 흥행으로 매출 급증',
        "Expect performance from signing HBM supply contract",
        "Increasing uncertainty due to increased currency volatility",
        "Sales Surge Due to the Success of New Products"]

# 문장별로 저장 리스트
rows = []

# 뉴스 문장을 하나씩 FinBERT로 감성분석
for n in news:
  result = finbert(n)[0]
  rows.append({'news': n,
               'label': result['label'],
               'score': result['score']})

# 분석 결과를 표 형태로 만들기
df = pd.DataFrame(rows)

# 감성별 개수 계산
count = Counter(df['label'])

# 긍정 뉴스 개수
pos = count['positive']

# 긍정 비율 계산
ratio = pos / len(df) * 100

# 결과 출력
print(df)
print(f'긍정 비율: {ratio:.2f}%')

                                                news     label     score
0                                HBM 공급 계약 체결로 실적 기대   neutral  0.891730
1                                 환율 변동성 확대로 불확실성 증가   neutral  0.869006
2                                     신제품 흥행으로 매출 급증   neutral  0.886919
3  Expect performance from signing HBM supply con...   neutral  0.707611
4  Increasing uncertainty due to increased curren...  negative  0.512139
5     Sales Surge Due to the Success of New Products  positive  0.947639
긍정 비율: 16.67%


2.3 파인튜닝 & PEFT / LoRA 개념

In [14]:
# !pip install peft transformers -q
# from peft import LoraConfig, get_peft_model
# from transformers import AutoModelForSequenceClassification

In [15]:
# !pip uninstall -y torchao gradio
# !pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1

In [16]:
# 실습 1 파인튜닝 & PEFT / LoRA 개념
# Klue-BERT 분류 모델에 LoRA 적용하고, 전체 파라미터 중 실제 학습되는 파라미터의 비율을 확인하시오

model = AutoModelForSequenceClassification.from_pretrained(
    'klue/bert-base', num_labels = 2) # 정상 : 0 스팸 : 1

lora = LoraConfig(r=8, lora_alpha=16,
                  target_modules=['query', 'value'],
                  lora_dropout=0.1)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 294,912 || all params: 110,913,794 || trainable%: 0.2659


In [17]:
# 실습 2 파인튜닝용 학습데이터를 딕셔너리 리스트로 준비하시오

train_data = [
    {'text': '오늘 종가와 시황 안내드립니다', 'label':0},
    {'text': '[광고] 급등주 무료로 받아가세요!', 'label':1},
    {'text': '배당금 지급 일정 공지입니다.', 'label':0},
    {'text': '수익률 300% 지금 클릭.', 'label':1}]

# 라벨 분포 확인(균형이 중요)

print(Counter(d['label'] for d in train_data))

Counter({0: 2, 1: 2})


In [18]:
# 실습 3 프롬프트 방식과 파인튜닝 방식의 장단점을 코드 주석으로 정리하시오

# 프롬프트 방식 - 즉시 사용, 모델 변경없음

def classify_by_prompt(text):
  p=f'''다음문자를 정상/스팸 중 하나로 분류해줘
        출력 규칙 :
        1. 정상 또는 스팸 중 하나만 출력
        2. 설명 금지
        문자 : {text}'''
  # openai api 호출
  r= client.chat.completions.create(
      model='gpt-4o-mini',
      messages=[{'role':'user','content':p}],
      temperature = 0,
      max_tokens = 5)
  # gpt 답변에서 실제 텍스트만 반환
  return r.choices[0].message.content

print("프롬프트 방식 결과 : ", classify_by_prompt("[광고]급등주 추천"))

프롬프트 방식 결과 :  스팸


In [19]:
# 프롬프트 방식과 파인튜닝 방식의 장단점을 코드 주석으로 정리하시오

# 기본모델
base_model = AutoModelForSequenceClassification.from_pretrained(
    'klue/bert-base', num_labels=2)

# LoRA 설정
lora = LoraConfig(r=8, lora_alpha=16,
                  target_modules=['query', 'value'],
                  lora_dropout=0.1)

# 기본모델에서 LoRA 적용
model = get_peft_model(base_model, lora)

# 실제 학습된 파라미터 비율 확인
model.print_trainable_parameters()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 294,912 || all params: 110,913,794 || trainable%: 0.2659


2.4 금융 문자 분류 경량 파인튜닝 실습

In [20]:
os.getcwd()
os.chdir('/content/drive/MyDrive/Colab Notebooks/09 LLM 기반 텍스트 분석')
os.getcwd()

'/content/drive/MyDrive/Colab Notebooks/09 LLM 기반 텍스트 분석'

In [24]:
# 실습 1 금융문자 분류 경량 파인튜닝
# 학습설정을 정의하고 Trainer로 학습

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,TrainingArguments, Trainer)
from datasets import load_dataset

# 모델과 토크나이저 불러오기
model_name = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# CSV 파일 불러오기
ds = load_dataset('csv', data_files='finance_sms.csv')['train']

# 토큰화
ds = ds.map(lambda e: tokenizer(e['text'], padding = 'max_length',
            truncation=True, max_length = 64), batched=True)

# 학습/평가 분리
split = ds.train_test_split(test_size=0.2, seed=42)
train_ds, eval_ds = split['train'], split['test']

print('학습 : ', len(train_ds), '/ 평가 : ', len(eval_ds))

# 학습 조건 설정 & 실행
args = TrainingArguments(
    output_dir = 'out',
    num_train_epochs = 3,
    per_device_train_batch_size = 8,    # 한번에 묶는 문자 데이터 수
    logging_steps=10,                   # 학습단계 10단계 진행할 때마다 손실값 등 출력
    eval_strategy = 'epoch')
# Trainer 생성
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds
)
# 학습 실행
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


학습 :  160 / 평가 :  40


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.005400,0.001254
2,0.000600,0.000371
3,0.000400,0.000309


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=60, training_loss=0.04350043432010959, metrics={'train_runtime': 493.1077, 'train_samples_per_second': 0.973, 'train_steps_per_second': 0.122, 'total_flos': 15786663321600.0, 'train_loss': 0.04350043432010959, 'epoch': 3.0})

In [25]:
# 실습 2 학습 후 평가
import numpy as np

# 정확도 계산 함수(eval_accuracy 생김)

def compute_metrics(p):
  preds = np.argmax(p.predictions, axis=1)
  return {'accuracy':(preds == p.label_ids).mean()}

trainer.compute_metrics = compute_metrics
after = round(trainer.evaluate()['eval_accuracy'],3)

before = 0.71
print('Before Accuracy : ', before)
print('After Accuracy : ', after)
print('개선폭 : ', f'+{round(after - before,2)}')


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Before Accuracy :  0.71
After Accuracy :  1.0
개선폭 :  +0.29


In [27]:
# 실습 3 학습한 모델로 새 금융 문자를 분류해보고 오분류 사례 확인

samples = ['오늘 보유 종목 시황 안내', '*급등임박* 지금 가입하세요']

import torch

for text in samples:
  inputs = tokenizer(text, return_tensors='pt')
  logits = model(**inputs).logits
  pred = torch.argmax(logits, dim=1).item()
  print(text, '->', '스팸' if pred == 1 else '정상')

오늘 보유 종목 시황 안내 -> 정상
*급등임박* 지금 가입하세요 -> 스팸


[미니 프로젝트] 금융뉴스 감성 대시보드 1안

In [50]:
# Step1. 데이터 준비
# 여러 종목의 뉴스 헤드라인을 종목명과 함께 리스트로 모은다

news_data = [
    # 삼성전자
    {'ticker': '삼성전자', 'news': '삼성전자, 3분기 영업이익 10조원 돌파...메모리 반등'},
    {'ticker': '삼성전자', 'news': '삼성전자, 파운드리 수율 부진으로 고객사 이탈 우려'},
    {'ticker': '삼성전자', 'news': '삼성전자, HBM4 양산 개시로 AI 반도체 경쟁력 강화'},
    {'ticker': '삼성전자', 'news': '삼성전자, 갤럭시 신모델 판매 호조로 실적 기대감 상승'},
    {'ticker': '삼성전자', 'news': '삼성전자, 스마트폰 사업부 수익성 둔화 우려 확대'},

    # SK하이닉스
    {'ticker': 'SK하이닉스', 'news': 'SK하이닉스, HBM 공급 계약 확대로 실적 기대감 상승'},
    {'ticker': 'SK하이닉스', 'news': 'SK하이닉스 ADR 프리미엄 과열 경고'},
    {'ticker': 'SK하이닉스', 'news': 'SK하이닉스, 낸드 가격 하락으로 수익성 압박'},
    {'ticker': 'SK하이닉스', 'news': 'SK하이닉스, 차세대 D램 개발 성공...기술 격차 확대'},

    # LG에너지솔루션
    {'ticker': 'LG에너지솔루션', 'news': 'LG에너지솔루션, 북미 배터리 공장 가동률 개선'},
    {'ticker': 'LG에너지솔루션', 'news': 'LG에너지솔루션, 전기차 수요 둔화로 목표주가 하향'},
    {'ticker': 'LG에너지솔루션', 'news': 'LG에너지솔루션, 유럽 배터리 공장 가동 중단 검토'},
    {'ticker': 'LG에너지솔루션', 'news': 'LG에너지솔루션, 원자재 가격 급등으로 원가 부담 확대'},
    {'ticker': 'LG에너지솔루션', 'news': 'LG에너지솔루션, 신규 수주 계약 체결로 매출 성장 기대'},
]

# 확인
for item in news_data:
    print(item['ticker'], '-', item['news'])

print('\n총 뉴스 건수 :', len(news_data))

삼성전자 - 삼성전자, 3분기 영업이익 10조원 돌파...메모리 반등
삼성전자 - 삼성전자, 파운드리 수율 부진으로 고객사 이탈 우려
삼성전자 - 삼성전자, HBM4 양산 개시로 AI 반도체 경쟁력 강화
삼성전자 - 삼성전자, 갤럭시 신모델 판매 호조로 실적 기대감 상승
삼성전자 - 삼성전자, 스마트폰 사업부 수익성 둔화 우려 확대
SK하이닉스 - SK하이닉스, HBM 공급 계약 확대로 실적 기대감 상승
SK하이닉스 - SK하이닉스 ADR 프리미엄 과열 경고
SK하이닉스 - SK하이닉스, 낸드 가격 하락으로 수익성 압박
SK하이닉스 - SK하이닉스, 차세대 D램 개발 성공...기술 격차 확대
LG에너지솔루션 - LG에너지솔루션, 북미 배터리 공장 가동률 개선
LG에너지솔루션 - LG에너지솔루션, 전기차 수요 둔화로 목표주가 하향
LG에너지솔루션 - LG에너지솔루션, 유럽 배터리 공장 가동 중단 검토
LG에너지솔루션 - LG에너지솔루션, 원자재 가격 급등으로 원가 부담 확대
LG에너지솔루션 - LG에너지솔루션, 신규 수주 계약 체결로 매출 성장 기대

총 뉴스 건수 : 14


In [51]:
# Step2. 감성분석 함수 만들기
import json
from openai import OpenAI

client = OpenAI()

def analyze_sentiment(item):
    """뉴스 1건(dict: ticker, news) -> {ticker, news, sentiment} 반환"""

    ticker = item['ticker']
    news = item['news']
    prompt = f'''너는 금융 애널리스트다.
아래 종목과 뉴스를 보고 투자 관점에서 감성을 판별하라.
반드시 아래 JSON 형식으로만 답하라. 다른 텍스트는 절대 포함하지 마라.
{{
  "ticker": "종목명",
  "sentiment": "긍정/부정/중립 중 하나"
}}

종목: {ticker}
뉴스: {news}'''

    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'})

    result = json.loads(r.choices[0].message.content)
    return result

# 테스트: 뉴스 1건만 넣어서 함수가 잘 동작하는지 확인
test_result = analyze_sentiment(news_data[0])
print(test_result)

{'ticker': '삼성전자', 'sentiment': '긍정'}


In [37]:
# Step3. 전체 뉴스 분석하기
import pandas as pd

# news_data의 모든 뉴스를 하나씩 analyze_sentiment 함수로 분석
results = [analyze_sentiment(item) for item in news_data]

# 결과를 DataFrame으로 변환
df = pd.DataFrame(results)

df

,ticker,sentiment
0,삼성전자,긍정
1,삼성전자,부정
2,삼성전자,긍정
3,삼성전자,긍정
4,삼성전자,부정
5,SK하이닉스,긍정
6,SK하이닉스,부정
7,SK하이닉스,부정
8,SK하이닉스,긍정
9,LG에너지솔루션,긍정


In [38]:
# 원문 뉴스도 함께 보고 싶을 때
df['news'] = [item['news'] for item in news_data]
df

,ticker,sentiment,news
0,삼성전자,긍정,"삼성전자, 3분기 영업이익 10조원 돌파...메모리 반등"
1,삼성전자,부정,"삼성전자, 파운드리 수율 부진으로 고객사 이탈 우려"
2,삼성전자,긍정,"삼성전자, HBM4 양산 개시로 AI 반도체 경쟁력 강화"
3,삼성전자,긍정,"삼성전자, 갤럭시 신모델 판매 호조로 실적 기대감 상승"
4,삼성전자,부정,"삼성전자, 스마트폰 사업부 수익성 둔화 우려 확대"
5,SK하이닉스,긍정,"SK하이닉스, HBM 공급 계약 확대로 실적 기대감 상승"
6,SK하이닉스,부정,SK하이닉스 ADR 프리미엄 과열 경고
7,SK하이닉스,부정,"SK하이닉스, 낸드 가격 하락으로 수익성 압박"
8,SK하이닉스,긍정,"SK하이닉스, 차세대 D램 개발 성공...기술 격차 확대"
9,LG에너지솔루션,긍정,"LG에너지솔루션, 북미 배터리 공장 가동률 개선"


In [39]:
# Step4. 종목별 집계하기

# 종목(ticker)별로 묶어서, sentiment가 '긍정'인 비율 계산
board = df.groupby('ticker')['sentiment'].apply(lambda x: (x == '긍정').mean())

board

,sentiment
ticker,
LG에너지솔루션,0.4
SK하이닉스,0.5
삼성전자,0.6


In [42]:
# Step5. 대시보드 완성 & 정렬

# 긍정 비율이 높은 순으로 정렬
board_sorted = board.sort_values(ascending=False)

print(board_sorted)

ticker
삼성전자        0.6
SK하이닉스      0.5
LG에너지솔루션    0.4
Name: sentiment, dtype: float64


In [41]:
# (선택) 보기 좋은 표로 다듬기
dashboard = board_sorted.reset_index()
dashboard.columns = ['종목', '긍정비율']
dashboard['긍정비율(%)'] = (dashboard['긍정비율'] * 100).round(1)

dashboard

,종목,긍정비율,긍정비율(%)
0,삼성전자,0.6,60.0
1,SK하이닉스,0.5,50.0
2,LG에너지솔루션,0.4,40.0


[미니 프로젝트] 금융뉴스 감성 대시보드 2안

In [53]:
# Step1. 데이터 준비
# 여러 종목의 뉴스 헤드라인을 종목명과 함께 리스트로 모은다
news_data = [
    'A전자 2분기 영업이익이 시장 전망치를 웃돌았다',
    'A전자 인공지능 반도체 수요 증가에 생산 확대를 검토한다',
    'A전자 원자재 가격 상승으로 하반기 수익성 둔화가 우려된다',
    'B바이오 신약 후보물질이 임상 3상 시험에서 유효성을 확인했다',
    'B바이오 미국 식품의약국에 신약 품목허가를 신청했다',
    'B바이오 연구개발비 증가로 영업손실이 확대됐다',
    'C자동차 전기차 해외 판매량이 전년보다 크게 증가했다',
    'C자동차 일부 차량에서 결함이 발견돼 자발적 리콜을 결정했다',
    'D건설 중동 지역에서 대규모 플랜트 공사를 수주했다',
    'D건설 공사비 상승과 분양 지연으로 실적 부담이 커졌다',
    'E금융 이자이익 증가로 분기 순이익이 개선됐다',
    'E금융 대출 연체율 상승으로 자산 건전성 우려가 제기됐다',
    'E금융 모바일 금융서비스 개편 계획을 발표했다'
]

print('\n총 뉴스 건수 :', len(news_data))


총 뉴스 건수 : 13


In [54]:
# Step2. 감성분석 함수 만들기 (문장에서 ticker를 직접 추출하는 버전)
import json
from openai import OpenAI

client = OpenAI()

def analyze_sentiment(news):
    prompt = f'''
    다음 금융 뉴스를 분석하여 JSON으로 출력하세요
    ticker: 종목명
    sentiment: 긍정/부정/중립 중 하나 선택
    뉴스: {news}'''

    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'})

    return json.loads(r.choices[0].message.content)

In [56]:
# Step3. 전체 뉴스 분석하기
import pandas as pd

results = [analyze_sentiment(n) for n in news_data]

df = pd.DataFrame(results)

df

,ticker,sentiment,뉴스
0,A전자,긍정,A전자 2분기 영업이익이 시장 전망치를 웃돌았다
1,A전자,긍정,A전자 인공지능 반도체 수요 증가에 생산 확대를 검토한다
2,A전자,부정,A전자 원자재 가격 상승으로 하반기 수익성 둔화가 우려된다
3,B바이오,긍정,B바이오 신약 후보물질이 임상 3상 시험에서 유효성을 확인했다
4,B바이오,긍정,B바이오 미국 식품의약국에 신약 품목허가를 신청했다
5,B바이오,부정,B바이오 연구개발비 증가로 영업손실이 확대됐다
6,C자동차,긍정,C자동차 전기차 해외 판매량이 전년보다 크게 증가했다
7,C자동차,부정,C자동차 일부 차량에서 결함이 발견돼 자발적 리콜을 결정했다
8,D건설,긍정,D건설 중동 지역에서 대규모 플랜트 공사를 수주했다
9,D건설,부정,D건설 공사비 상승과 분양 지연으로 실적 부담이 커졌다


In [48]:
# Step4
board = df.groupby('ticker')['sentiment'].apply(lambda x: (x == '긍정').mean())
board

,sentiment
ticker,
A전자,0.666667
B바이오,0.666667
C자동차,0.500000
D건설,0.500000
E금융,0.666667


In [49]:
# Step5
board_sorted = board.sort_values(ascending=False) # 오름차순이 기본값이라 False
print(board_sorted)

ticker
A전자     0.666667
B바이오    0.666667
E금융     0.666667
C자동차    0.500000
D건설     0.500000
Name: sentiment, dtype: float64
